# Nemotron LoRA v53 — Adapter Training (No pip installs)
Pre-attached Nemotron-4-15B. Uses only torch+transformers+peft+datasets.
Generates reasoning traces, trains LoRA, outputs submission.zip

In [ ]:
import json
import os
import zipfile

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)


# ── PATHS ────────────────────────────────────────
COMP = "nvidia-nemotron-model-reasoning-challenge"
OUT = "/kaggle/working/output"
TRAIN = f"/kaggle/input/{COMP}/train.csv"
NEMO = "/kaggle/input/models/nvidia/llama/nemotron-4-15b-instruct/1"
os.makedirs(OUT, exist_ok=True)

# Verify Nemotron exists
if not os.path.exists(NEMO):
    for r, d, f in os.walk("/kaggle/input"):
        if "config.json" in f:
            print("Found model at", r)
            NEMO = r
            break


# ── SYMBOLIC SOLVER ──────────────────────────────
def pairs(s):
    r = []
    for ln in s.split(chr(10)):
        if " -> " in ln and not ln.startswith(("Here", "Now")):
            a, b = ln.split(" -> ", 1)
            r.append((a.strip(), b.strip()))
        elif " becomes " in ln and not ln.startswith("Now"):
            a, b = ln.split(" becomes ", 1)
            r.append((a.strip(), b.strip()))
        elif "=" in ln and not ln.startswith(("Here", "Now", "For", "The")):
            a, b = ln.split("=", 1)
            r.append((a.strip(), b.strip()))
    return r


def kind(s):
    s = s.lower()
    if "bit" in s:
        return "bit"
    if any(x in s for x in ("gravit", "fall", "drop")):
        return "grav"
    if "unit" in s:
        return "unit"
    if "equation" in s or "rule" in s:
        return "eq"
    if "roman" in s or "numeral" in s:
        return "rom"
    if any(x in s for x in ("cipher", "encrypt", "decrypt")):
        return "enc"
    if "sum" in s or "add" in s:
        return "sum"
    if "product" in s or "multiply" in s:
        return "prod"
    return "unk"


def solve(p):
    ex, ti = [], ""
    if any(x in p for x in (" -> ", "Test input", " becomes ", "=")):
        ex = pairs(p)
        if "Test input" in p:
            ti = p.split("Test input")[-1].split(":")[-1].strip()
        else:
            ti = ex[-1][0] if ex else ""
    else:
        ti = p
    c = kind(p)
    if c == "grav" and ex:
        ts, ds = [], []
        for a, b in ex:
            try:
                t = float(__import__("re").search(r"([0-9.]+)", a).group(1))
                d = float(__import__("re").search(r"([0-9.]+)", b).group(1))
                ts.append(t)
                ds.append(d)
            except:
                pass
        if len(ts) < 2:
            return ""
        xs = [0.5 * t * t for t in ts]
        sx = sum(d * x for d, x in zip(ds, xs))
        s2 = sum(x * x for x in xs)
        g = sx / s2 if s2 else 0
        bg, be = g, 1e9
        for g0 in [g] + [2 * d / (t * t) for t, d in zip(ts, ds) if t]:
            for step in [0.01, 0.005, 0.002, 0.001]:
                for off in range(-5, 6):
                    gg = g0 + off * step
                    e = sum((0.5 * gg * tt * tt - d) ** 2 for tt, d in zip(ts, ds))
                    if e < be:
                        be, bg = e, gg
        try:
            tv = float(__import__("re").search(r"([0-9.]+)", ti).group(1))
        except:
            return ""
        return str(0.5 * bg * tv * tv)
    elif c == "unit":
        M = {
            "inch": 0.0254,
            "foot": 0.3048,
            "yard": 0.9144,
            "mile": 1609.34,
            "cm": 0.01,
            "m": 1,
            "km": 1000,
            "mm": 0.001,
        }
        for a, b in ex:
            if a.lower() in M and b.lower() in M:
                return str(float(ti) * M.get(a.lower(), 1) / M.get(b.lower(), 1))
        return ""
    elif c == "bit":
        for a, b in ex:
            if set(a) <= {"0", "1"} and set(b) <= {"0", "1"}:
                try:
                    return bin(int(ti, 2) & int(a, 2))[2:]
                except:
                    pass
        return ""
    elif c == "rom":
        R = {
            "I": 1,
            "V": 5,
            "X": 10,
            "L": 50,
            "C": 100,
            "D": 500,
            "M": 1000,
            "IV": 4,
            "IX": 9,
            "XL": 40,
            "XC": 90,
            "CD": 400,
            "CM": 900,
        }

        def rv(s):
            i = t = 0
            while i < len(s):
                if i + 1 < len(s) and s[i : i + 2] in R:
                    t += R[s[i : i + 2]]
                    i += 2
                else:
                    t += R.get(s[i], 0)
                    i += 1
            return t

        for a, b in ex:
            return str(rv(ti)) if b == str(rv(a)) else ""
        return ""
    elif c == "eq":
        if not ex:
            return ""
        xs, ys = [], []
        for a, b in ex:
            try:
                xa = float(a)
                ya = float(__import__("re").sub(r"[^0-9.-]", "", b))
                xs.append(xa)
                ys.append(ya)
            except:
                pass
        n = len(xs)
        if n < 2:
            return ""
        sx = sum(xs)
        sy = sum(ys)
        sxx = sum(x * x for x in xs)
        sxy = sum(x * y for x, y in zip(xs, ys))
        d0 = n * sxx - sx * sx
        m = (n * sxy - sx * sy) / d0 if d0 else 0
        b0 = (sy - sx * m) / n if n else sum(ys) / n
        try:
            tx = float(ti)
        except:
            return ""
        return str(m * tx + b0)
    elif c == "sum":
        try:
            return str(sum(float(a) + float(b) for a, b in ex))
        except:
            return ""
    elif c == "prod":
        try:
            p = 1
            for a, b in ex:
                p *= float(a)
            return str(float(ti) * p)
        except:
            return ""
    return ""


# ── GENERATE TRACES ──────────────────────────────
df = __import__("pandas").read_csv(TRAIN)
print(f"Loaded {len(df)} rows")
traces = []
for _, row in df.iterrows():
    p = str(row.get("prompt", row.get("question", "")))
    a = str(row.get("answer", ""))
    s = solve(p)
    tr = f"Problem: {p}\n\nI need to analyze this step by step.\n"
    if s:
        tr += f"From the pattern, {s}\n"
    tr += f"Therefore, answer is \n\boxed{{{a}}}."
    traces.append(tr)

# ── TOKENIZER + MODEL ───────────────────────────
tok = AutoTokenizer.from_pretrained(NEMO)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(NEMO, torch_dtype=torch.bfloat16, device_map="auto")
print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B params")

# ── LoRA ───────────────────────────────────────
lora = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

# ── DATASET ────────────────────────────────────
ds = Dataset.from_dict({"text": traces})


def tk(b):
    o = tok(b["text"], truncation=True, padding="max_length", max_length=512, return_tensors=None)
    o["labels"] = o["input_ids"].copy()
    return o


tok_ds = ds.map(tk, batched=True, remove_columns=["text"])

# ── TRAINER ────────────────────────────────────
args = TrainingArguments(
    output_dir=OUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="epoch",
    remove_unused_columns=False,
    report_to=["none"],
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True),
)
print("\nTraining...")
trainer.train()

# ── SAVE ───────────────────────────────────────
AD = os.path.join(OUT, "adapter")
model.save_pretrained(AD)
tok.save_pretrained(AD)
with open(os.path.join(AD, "adapter_config.json")) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = NEMO
with open(os.path.join(AD, "adapter_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

# ── ZIP ────────────────────────────────────────
sub = os.path.join(OUT, "submission.zip")
with zipfile.ZipFile(sub, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in ["adapter_config.json", "adapter_model.safetensors"]:
        p = os.path.join(AD, fn)
        if os.path.exists(p):
            z.write(p, arcname=fn)
            print("Added", fn)
print("\n=== SUBMISSION ===")
print(f"File: {sub}")
print(f"Size: {os.path.getsize(sub) / 1024 / 1024:.1f} MB")